# Verifier-driven MCRDR learning: VAREX trial

This notebook learns from a set of VAREX documents one at a time, then tests the result on different documents.

**Learning** uses the first `LEARN_DOCUMENTS` *held-out* documents (never the 30 dev documents of the earlier
pilot), `BATCH_SIZE` documents per round (default 1). Each round:
1. The **worker** (Qwen3.7 Flash) does the document with the tree learned so far (the original MCRDR tree at first).
2. If it passed, its advice becomes "worked fine" examples (cornerstones) in the tree.
3. If it failed, the **inspector** (Qwen3.7 Plus) finds the cause and must prove it by fixing the answer and
   re-scoring it with the official scorer. A proven, skill-related cause becomes a proposed MCRDR rule.
4. The **checker** tests the rule: the tree engine refuses it if it would change the advice for any "worked fine"
   example, and the worker is re-run on the failed document (must now pass) and a few passing ones (must still pass).
5. A kept rule is in the tree for every later document, so a problem it fixes is not learned twice.

**Testing**: the final learned tree runs on the 30 dev documents, unseen during learning, and is compared with the
pilot deck's numbers for the original tree on those same documents (and, optionally, a fresh re-run of it).

All models run on OpenRouter (provider `alibaba`). The **judge** that answers learned rule conditions (true / false / unknown) is Qwen3.7 Flash; it reads
the worker's starting materials (task, requested schema, the PDF's text) and the worker's recent messages.
Models and limits are in `examples/learning/varex.learner.json`; the worker's settings match the pilot deck.

**Cost.** Paid cells are marked. Rough guesses, not measurements: about $0.02 per worker attempt (so about $0.60
for 30 documents once); the inspector up to about $0.30 per failed document; re-testing about $0.20 per rule.

**How to run it**

1. In VS Code, open this notebook with the **Google Colab** extension: **Select Kernel → Colab**, start a server
   (a CPU runtime is enough).
2. Run the cells in order. Each paid cell can be re-run after a disconnect: finished work is kept and resumed.

Results stay on the runtime under `/content/rdr-skills-mcrdr/experiments/`; the last cell downloads a zip.
Diagnoses contain the correct answers and episodes contain document text, so keep them private.

In [ ]:
# Settings. Change them only deliberately; a new RUN_NAME starts a fresh trial.
BRANCH = "verifier-mcrdr"
WORKDIR = "/content/rdr-skills-mcrdr"
EXPERIMENT = "qwen-dev30"  # examples/agent/varex.mcrdr.qwen-dev30.json (worker model and limits)
LEARN_DOCUMENTS = 30  # learning set: the first N held-out documents, never the 30 dev documents
BATCH_SIZE = 1  # documents per learning round; 1 = one at a time, LEARN_DOCUMENTS = all at once
LEARN_REPLICATES = 1  # worker attempts per learning document; more attempts give more failures to learn from
RUN_NAME = "learn-heldout30"
TEST_RUNS = 1  # runs of the learned tree on the 30 dev documents (the pilot deck used 3)
RERUN_BASE = False  # also re-run the original tree on the 30 dev documents, same cost as a learned run

## 1. Model key

In [ ]:
import getpass
import os

if not os.environ.get("MODEL_API_KEY"):
    key = None
    try:  # Colab Secrets, if this notebook has been granted access to a MODEL_API_KEY secret
        from google.colab import userdata
        key = userdata.get("MODEL_API_KEY")
    except Exception:
        pass
    os.environ["MODEL_API_KEY"] = key or getpass.getpass("OpenRouter key (hidden, not saved): ")
print("MODEL_API_KEY is set" if os.environ.get("MODEL_API_KEY") else "MODEL_API_KEY is missing")

## 2. Setup script

Fetches `scripts/setup_mcrdr_colab.py` from the branch. It needs only the standard library.

In [ ]:
import importlib.util
import sys
import urllib.request
from pathlib import Path

SCRIPT = Path("/content/setup_mcrdr_colab.py")
urllib.request.urlretrieve(
    f"https://raw.githubusercontent.com/dmw2101/RDR-skills-/{BRANCH}/scripts/setup_mcrdr_colab.py", SCRIPT)
spec = importlib.util.spec_from_file_location("setup_mcrdr_colab", SCRIPT)
setup_mcrdr_colab = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = setup_mcrdr_colab
spec.loader.exec_module(setup_mcrdr_colab)
EXPERIMENTS_DIR = Path(WORKDIR) / "experiments"
LEARNER_CONFIG = EXPERIMENTS_DIR / "configs" / f"{RUN_NAME}.learner.json"
LEARNING_DIR = EXPERIMENTS_DIR / "learning" / RUN_NAME
PYTHON = Path(WORKDIR) / ".venv" / "bin" / "python"


def setup_learning():
    """Set up (idempotently) and check everything; write the run and learner configs. No model calls."""
    args = ["--experiment", EXPERIMENT, "--heldout-batch", str(LEARN_DOCUMENTS),
            "--replicates", str(LEARN_REPLICATES), "--conditions", "reduced", "--run-name", RUN_NAME,
            "--workdir", WORKDIR, "--branch", BRANCH, "--learn", "--batch-size", str(BATCH_SIZE)]
    if setup_mcrdr_colab.main(args):
        raise RuntimeError(f"{RUN_NAME} setup stopped; read the message above, fix it, and rerun this cell")
    print(f"\n{RUN_NAME}: set up and checked")


def learn(stage):
    """Run a trace_learner command; learning saves every step and resumes after an interruption."""
    if not LEARNER_CONFIG.exists():
        raise RuntimeError("run the setup cell first; it writes the learner config")
    setup_mcrdr_colab.Runner().run([PYTHON, "-m", "trace_learner", stage, "--config", LEARNER_CONFIG],
                                   cwd=Path(WORKDIR))

## 3. Set up and check, without model calls

Installs the PDF tools and Python packages, clones the branch, downloads the learning documents, freezes the
original PDF skill tree, starts the MCRDR skill service (port 8766), writes the run and learner configs, and runs
`doctor`. Nothing is spent here.

In [ ]:
setup_learning()

## 4. Learn, one document at a time (paid)

Prints one block per round: the document, whether the worker passed, the inspector's verdict on a failure, each
proposed rule and the checker's decision, and the tree version the next document will use. If the runtime
disconnects, run section 1 and then this cell again: finished rounds and model calls are kept.

In [ ]:
learn("learn")

## 5. Results of learning

In [ ]:
from IPython.display import Markdown, display

learn("report")
display(Markdown((LEARNING_DIR / "report.md").read_text(encoding="utf-8")))

## 6. Test the learned tree on the 30 dev documents (paid)

Runs the final learned tree on the 30 dev documents, which learning never saw, `TEST_RUNS` times, with the
same worker settings as the pilot deck. The judge answers the learned conditions before every step. With
`RERUN_BASE = True` the original tree is re-run alongside, so both are measured on the same day.

In [ ]:
import json

state = json.loads((LEARNING_DIR / "state.json").read_text(encoding="utf-8"))
rounds = json.loads((LEARNING_DIR / "rounds.json").read_text(encoding="utf-8"))
if state["round"] * BATCH_SIZE < len(state["task_order"]):
    raise RuntimeError("learning has not finished; run section 4 again to continue it")
if not sum(len(item["accepted"]) for item in rounds.values()):
    raise RuntimeError("the checker kept no rule, so the tree gives the original advice (see section 5)")
VERSION = state["current_version"]
SKILL_VERSION = LEARNING_DIR / "versions" / f"v{VERSION}" / "pdf"
LEARNED_RUNS = [f"{RUN_NAME}-v{VERSION}-dev30-run{n}" for n in range(1, TEST_RUNS + 1)]
BASE_RUNS = [f"base-dev30-run{n}" for n in range(1, TEST_RUNS + 1)] if RERUN_BASE else []


def evaluate(name, *extra):
    """Set up, run, inspect and summarize one reduced-arm run on the 30 dev documents."""
    args = ["--experiment", EXPERIMENT, "--conditions", "reduced", "--run-name", name,
            "--workdir", WORKDIR, "--branch", BRANCH, "--run", *extra]
    if (EXPERIMENTS_DIR / "results" / name).exists():
        args.append("--resume")
    if setup_mcrdr_colab.main(args):
        raise RuntimeError(f"{name} stopped; read the message above, fix it, and rerun this cell")


for learned, base in zip(LEARNED_RUNS, BASE_RUNS or [None] * len(LEARNED_RUNS)):
    evaluate(learned, "--skill-version", str(SKILL_VERSION))
    if base:
        evaluate(base)
print(f"Tested v{VERSION}:", *LEARNED_RUNS, *BASE_RUNS)

## 7. Compare with the original tree

The pilot deck measured the original tree (same advice as today's MCRDR tree) on these 30 documents, 3 runs, with
the same worker settings. Only its totals are available, so the comparison is on totals; a re-run of the original
tree (`RERUN_BASE`) also allows a document-by-document comparison in the report below. A single run varies: the
deck's own runs scored 85.7% to 94.0%.

In [ ]:
import subprocess

from IPython.display import Markdown, display

# Pilot deck (VAREX-skill-guidance-pilot-v2): the reduced arm, i.e. the original tree, 3 runs x 30 dev documents.
DECK_REDUCED = {"field_exact_match": {"Run 1": 91.70, "Run 2": 85.66, "Run 3": 93.96, "Pooled": 90.44},
                "fields_correct": 719, "fields_total": 795, "completed": 87, "perfect_documents": 68, "episodes": 90}

COMPARISON = LEARNING_DIR / "comparison"
COMPARISON.mkdir(exist_ok=True)
series = {}
if BASE_RUNS:
    series["Original tree (re-run)"] = {f"Run {n}": [str(EXPERIMENTS_DIR / "results" / name)]
                                        for n, name in enumerate(BASE_RUNS, 1)}
series[f"Learned tree v{VERSION}"] = {f"Run {n}": [str(EXPERIMENTS_DIR / "results" / name)]
                                      for n, name in enumerate(LEARNED_RUNS, 1)}
SPEC = COMPARISON / "report-spec.json"
SPEC.write_text(json.dumps({"series": series}, indent=2), encoding="utf-8")
result = subprocess.run([str(PYTHON), "-m", "skill_agent", "report", "--spec", str(SPEC)], cwd=WORKDIR,
                        capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr or result.stdout)
report = json.loads((COMPARISON / "report" / "report.json").read_text(encoding="utf-8"))


def totals(stats):
    return (f"{stats['field_exact_match'] * 100:.2f}%", f"{stats['fields_correct']} / {stats['fields_total']}",
            f"{stats['perfect_documents']} / {stats['episodes']}", f"{stats['completed']} / {stats['episodes']}")


rows = [("Original tree, pilot deck (3 runs)", f"{DECK_REDUCED['field_exact_match']['Pooled']:.2f}%",
         f"{DECK_REDUCED['fields_correct']} / {DECK_REDUCED['fields_total']}",
         f"{DECK_REDUCED['perfect_documents']} / {DECK_REDUCED['episodes']}",
         f"{DECK_REDUCED['completed']} / {DECK_REDUCED['episodes']}")]
for name, data in report["series"].items():
    rows.append((f"{name}, pooled ({len(data['runs'])} run(s))", *totals(data["pooled"]["conditions"]["reduced"])))
    for run, stats in data["runs"].items():
        rows.append((f"{name}, {run}", *totals(stats["conditions"]["reduced"])))
table = ["| Tree | Exact-match fields | Correct fields | Perfect documents | Completed |", "|---|---|---|---|---|"]
table += ["| " + " | ".join(row) + " |" for row in rows]
table.append("")
table.append("Pilot deck single runs: " + ", ".join(
    f"{run} {value:.2f}%" for run, value in DECK_REDUCED["field_exact_match"].items() if run != "Pooled") + ".")
display(Markdown("\n".join(table)))
display(Markdown((COMPARISON / "report" / "report.md").read_text(encoding="utf-8")))

## 8. Download the results

In [ ]:
import zipfile

ARCHIVE = Path(f"/content/{RUN_NAME}.zip")
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(EXPERIMENTS_DIR.rglob("*")):
        relative = path.relative_to(EXPERIMENTS_DIR)
        keep = ((relative.parts[0] == "results" and len(relative.parts) > 1
                 and relative.parts[1].startswith((RUN_NAME, "base-dev30-run")))
                or relative.parts[:2] == ("learning", RUN_NAME)
                or relative.parts[0] in ("configs", "logs") or relative.name.startswith("pdf-snapshot-"))
        if path.is_file() and keep:
            archive.write(path, relative.as_posix())
print(f"Wrote {ARCHIVE} ({ARCHIVE.stat().st_size / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(str(ARCHIVE))
except Exception:
    print("Automatic download is unavailable here; fetch the file from the runtime's file browser.")